# 03 材料描述符：从结构到特征矩阵

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/03_material_descriptors.ipynb)

## Learning objectives
理解 representation 决定模型能看见什么；区分 composition、geometry、chemistry、topology 四类 COF 特征。

In [ ]:
!pip -q install pymatgen matminer

## 1. 三个表示层次
**Composition descriptors**：只知道有哪些元素及比例。

**Structure/geometric descriptors**：加入晶格、局部环境、密度、孔径、表面积、void fraction 等。

**Learned representation**：GNN 等直接从 atomic graph 学习。

不存在对所有任务都最好的表示。预测 CO₂ uptake、band gap、diffusion coefficient 所需的信息并不相同。

In [ ]:
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
import pandas as pd
formulas=['C6H6','C6H4N2','C6H4O2','C6H2F4','C6H4S']
df=pd.DataFrame({'composition':[Composition(x) for x in formulas]})
feat=ElementProperty.from_preset('magpie')
out=feat.featurize_dataframe(df,col_id='composition',ignore_errors=True)
print('Feature count:',out.shape[1]-1)
display(out.iloc[:,:8])

## 2. 为什么通用元素描述符对 COF 不够？
Magpie 一类特征不知道：
- largest cavity / limiting pore diameter；
- accessible surface area / void fraction；
- topology；
- linkage chemistry；
- functional-group position；
- interlayer stacking/slip；
- hydrated pore environment。

因此一个实用 COF baseline 往往是 **chemical descriptors + pore/geometric descriptors + structural metadata** 的组合。

In [ ]:
cof_features=pd.DataFrame({
 'COF':['A','B','C'],
 'density_g_cm3':[0.55,0.78,0.61],
 'largest_pore_A':[28.0,17.5,22.1],
 'void_fraction':[0.72,0.51,0.65],
 'N_fraction':[0.08,0.13,0.05],
 'O_fraction':[0.10,0.08,0.15],
 'linkage':['imine','beta-ketoenamine','imine'],
 'topology':['hcb','hcb','sql']
})
cof_features

## 3. Feature 必须与目标物理过程对应
以 CO₂ transport 为例，孔径和 void fraction 描述几何通道；N/O/官能团描述局部相互作用；stacking 描述层间瓶颈；湿态输运还可能需要 hydration/H-bond descriptors。

**特征越多不一定越好。** 应先问：这个变量在物理上为什么可能影响 target？

## Exercises
设计适合以下三个 target 的特征，并说明物理理由：
1. CO₂ adsorption；
2. CO₂ diffusion；
3. electronic band gap。

要求把特征分为 geometric、chemical、topological/structural、energetic 四类，并指出哪些特征可能造成 data leakage。

### Take-home message
Representation 是材料知识进入机器学习模型的接口。